<a href="https://colab.research.google.com/github/ShaughanR/Wine-Neural-Network/blob/main/ML_Assignment3_ShaughanReddy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np
import random

#cell with code for question one

#[178x13] matrix, 3 classes, 2 hidden layers, so 13x4x3x3 Neural Network
# import some data
wine_data = load_wine()

#retrieve the data
X = wine_data.data
y = wine_data.target

#Split data in training and testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3)

#normalize data to 0-1 using min-max
min = np.min(X_train, axis=0)
max = np.max(X_train, axis=0)
X_train = (X_train - min) / (max - min)
X_test = (X_test - min) / (max - min)








#method to perform one forward pass
def ForwardPass(X_train, y_train, first_hidden_layer_weight_vector, first_hidden_layer_bias_vector,
                second_hidden_layer_weight_vector, second_hidden_layer_bias_vector,
                output_layer_weight_vector, output_layer_bias_vector, row, index):

  #forward propagation using ReLu
    #first hidden layer
    first_hidden_layer_input_weight = np.dot(row, first_hidden_layer_weight_vector)
    first_hidden_layer_with_bias = np.add(first_hidden_layer_input_weight, first_hidden_layer_bias_vector)
    first_hidden_layer_activated = np.maximum(0, first_hidden_layer_with_bias)

    #second hidden layer
    second_hidden_layer_input_weight = np.dot(first_hidden_layer_activated, second_hidden_layer_weight_vector)
    second_hidden_layer_with_bias = np.add(second_hidden_layer_input_weight, second_hidden_layer_bias_vector)
    second_hidden_layer_activated = np.maximum(0, second_hidden_layer_with_bias)

    #output layer
    output_layer_input_weight = np.dot(second_hidden_layer_activated, output_layer_weight_vector)
    output_layer_with_bias = np.add(output_layer_input_weight, output_layer_bias_vector)

    #perform softmax on output layer
    exp_vector = np.exp(output_layer_with_bias)
    output_layer_probabilities = exp_vector / np.sum(exp_vector)

    #one-hot encoding
    correct_class = [0, 0, 0]
    correct_class[y_train[index]] = 1
    correct_class = np.array(correct_class)

    #compute loss
    loss = np.sum(-correct_class * np.log(output_layer_probabilities))
    loss_error = output_layer_probabilities - correct_class

    return (first_hidden_layer_input_weight, first_hidden_layer_with_bias, first_hidden_layer_activated,
          second_hidden_layer_input_weight, second_hidden_layer_with_bias, second_hidden_layer_activated,
          output_layer_input_weight, output_layer_with_bias, output_layer_probabilities, loss_error, row)







#method to perform backpropagation once
def BackPropagation(first_hidden_layer_with_bias, first_hidden_layer_activated,
                    first_hidden_layer_weight_vector, first_hidden_layer_bias_vector,
                    second_hidden_layer_with_bias, second_hidden_layer_activated,
                    second_hidden_layer_weight_vector, second_hidden_layer_bias_vector,
                    output_layer_bias_vector, output_layer_weight_vector, loss_error, row, learning_rate):

    #start backpropogating weight/bias updates
    #backprop for output layer
    output_layer__weights_gradient = np.outer(second_hidden_layer_activated, loss_error)
    output_layer_bias_gradient = loss_error
    output_layer_weight_vector -= learning_rate * output_layer__weights_gradient
    output_layer_bias_vector -= learning_rate * output_layer_bias_gradient



    #backprop for second hidden layer
    second_hidden_layer_error = np.dot(loss_error, output_layer_weight_vector)

    #compute second hidden layer's error using ReLu derivative
    ReLu_derivative_values = []
    #flatten array for indicing
    second_hidden_layer_with_bias_flatten = second_hidden_layer_with_bias.flatten()
    for element in range(len(second_hidden_layer_with_bias_flatten)):
      if second_hidden_layer_with_bias_flatten[element] <= 0:
        ReLu_derivative_values.append(0)
      else:
        ReLu_derivative_values.append(1)

    #calculate error value for second hidden layer
    second_hidden_layer_error = np.multiply(second_hidden_layer_error, ReLu_derivative_values)

    #find gradients and update weights and biases
    second_hidden_layer_weights_gradient = np.outer(first_hidden_layer_activated, second_hidden_layer_error)
    second_hidden_layer_bias_gradient = second_hidden_layer_error
    second_hidden_layer_weight_vector -= learning_rate * second_hidden_layer_weights_gradient
    second_hidden_layer_bias_vector -= learning_rate * second_hidden_layer_bias_gradient


    #backprop for first hidden layer
    first_hidden_layer_error = np.dot(second_hidden_layer_error, second_hidden_layer_weight_vector.T)

    #compute second hidden layer's error using ReLu derivative
    ReLu_derivative_values = []
    #flatten array for indicing
    first_hidden_layer_with_bias_flatten = first_hidden_layer_with_bias.flatten()
    for element in range(len(first_hidden_layer_with_bias_flatten)):
      if first_hidden_layer_with_bias_flatten[element] <= 0:
        ReLu_derivative_values.append(0)
      else:
        ReLu_derivative_values.append(1)

    #calculate error value for first hidden layer
    first_hidden_layer_error = np.multiply(first_hidden_layer_error, ReLu_derivative_values)

    #find gradients and update weights and biases
    first_hidden_layer_weights_gradient = np.outer(row, first_hidden_layer_error)
    first_hidden_layer_bias_gradient = first_hidden_layer_error
    first_hidden_layer_weight_vector -= learning_rate * first_hidden_layer_weights_gradient
    first_hidden_layer_bias_vector -= learning_rate * first_hidden_layer_bias_gradient

    return (first_hidden_layer_weight_vector, first_hidden_layer_bias_vector,
          second_hidden_layer_weight_vector, second_hidden_layer_bias_vector,
          output_layer_weight_vector, output_layer_bias_vector)








#initialize weight vectors for each layer
first_hidden_layer_weight_vector = np.random.rand(13, 4)
second_hidden_layer_weight_vector = np.random.rand(4, 3)
output_layer_weight_vector = np.random.rand(3, 3)

#initialize bias vectors for each layer
first_hidden_layer_bias_vector = np.random.rand(4)
second_hidden_layer_bias_vector = np.random.rand(3)
output_layer_bias_vector = np.random.rand(3)

#learning rate and epochs
learning_rate = 0.001
epochs = 1000
current_epoch = 0



#loop through each epoch
while(current_epoch < epochs):
    current_epoch += 1

    #loop through each sample in training data and update weights/biases using
    #stochastic gradient descent
    for index, row in enumerate(X_train):
      #call forward pass method on each sample
      first_hidden_layer_input_weight, first_hidden_layer_with_bias, first_hidden_layer_activated, \
      second_hidden_layer_input_weight, second_hidden_layer_with_bias, second_hidden_layer_activated, \
      output_layer_input_weight, output_layer_with_bias, output_layer_probabilities, loss_error, row = ForwardPass(
          X_train, y_train, first_hidden_layer_weight_vector, first_hidden_layer_bias_vector,
          second_hidden_layer_weight_vector, second_hidden_layer_bias_vector,
          output_layer_weight_vector, output_layer_bias_vector, row, index)


      #call backpropogate method on each sample after finding loss error value
      first_hidden_layer_weight_vector, first_hidden_layer_bias_vector, \
      second_hidden_layer_weight_vector, second_hidden_layer_bias_vector, \
      output_layer_weight_vector, output_layer_bias_vector = BackPropagation(
          first_hidden_layer_with_bias, first_hidden_layer_activated,
          first_hidden_layer_weight_vector, first_hidden_layer_bias_vector,
          second_hidden_layer_with_bias, second_hidden_layer_activated,
          second_hidden_layer_weight_vector, second_hidden_layer_bias_vector,
          output_layer_bias_vector, output_layer_weight_vector, loss_error, row, learning_rate)


#loop through each sample in testing data
test_samples_probabilites = []
for index, row in enumerate(X_test):
  #call forward pass method on each sample
  first_hidden_layer_input_weight, first_hidden_layer_with_bias, first_hidden_layer_activated, \
  second_hidden_layer_input_weight, second_hidden_layer_with_bias, second_hidden_layer_activated, \
  output_layer_input_weight, output_layer_with_bias, output_layer_probabilities, loss_error, row = ForwardPass(
    X_test, y_test, first_hidden_layer_weight_vector, first_hidden_layer_bias_vector,
    second_hidden_layer_weight_vector, second_hidden_layer_bias_vector,
    output_layer_weight_vector, output_layer_bias_vector, row, index)

  #save probabiliy values for each sample
  test_samples_probabilites.append(output_layer_probabilities)

#use argmax to convert probability to class label
test_samples_predicted_class_labels = []
for sample in test_samples_probabilites:
  test_samples_predicted_class_labels.append(np.argmax(sample))

#compute classification accuracy of predicted class labels and
#correct class labels on testing data
correct_predictions_accuracy = np.mean(test_samples_predicted_class_labels == y_test)

#output classification accuracy value
print("Classification Accuracy: " + str(correct_predictions_accuracy))





















Classification Accuracy: 0.9074074074074074


In [ ]:
import numpy as np
from tensorflow import keras
from keras.datasets import cifar10
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
from tensorflow.keras import layers, models
from sklearn.metrics import f1_score

#cell with code for question two


#combine data to get correct train/test split
(X_train, y_train), (X_test, y_test) = cifar10.load_data()
X_data = np.concatenate((X_train, X_test))
y_data = np.concatenate((y_train, y_test))

#normalize data
X_data = X_data.astype('float32') / 255

#one-hot encode target data
y_data = to_categorical(y_data)

#split data
X_train, X_test, y_train, y_test = train_test_split(X_data, y_data, test_size=0.3)


#create model architecture and add convolutional and pooling layers
model = models.Sequential()
model.add(layers.Conv2D(32, (3, 3), activation='relu', input_shape=(32, 32, 3)))
model.add(layers.MaxPooling2D((2, 2)))
model.add(layers.Conv2D(64, (3, 3), activation='relu'))
model.add(layers.MaxPooling2D((2, 2)))
model.add(layers.Conv2D(64, (3, 3), activation='relu'))

#flatten data and combine features to make class predictions
model.add(layers.Flatten())
model.add(layers.Dense(64, activation='relu'))
model.add(layers.Dense(10, activation='softmax'))

#compile the model
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

#train model
model.fit(X_train, y_train, epochs=10, batch_size=64, validation_split=0.1)

#compute accuracy of model using testing data on trained model
test_loss, test_accuracy = model.evaluate(X_test, y_test)
print("Test Accuracy: " + str(test_accuracy))

#compute f1 score of model using testing data
predictions = model.predict(X_test)
predicted_classes = np.argmax(predictions, axis=1)
correct_classes = np.argmax(y_test, axis=1)
f1 = f1_score(correct_classes, predicted_classes, average='macro')
print("F1 Score: " + str(f1))









/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/15
591/591 ━━━━━━━━━━━━━━━━━━━━ 45s 73ms/step - accuracy: 0.3013 - loss: 1.8823 - val_accuracy: 0.5143 - val_loss: 1.3608
Epoch 2/15
591/591 ━━━━━━━━━━━━━━━━━━━━ 44s 74ms/step - accuracy: 0.5273 - loss: 1.3210 - val_accuracy: 0.5660 - val_loss: 1.2302
Epoch 3/15
591/591 ━━━━━━━━━━━━━━━━━━━━ 43s 73ms/step - accuracy: 0.5882 - loss: 1.1547 - val_accuracy: 0.6181 - val_loss: 1.0998
Epoch 4/15
591/591 ━━━━━━━━━━━━━━━━━━━━ 43s 72ms/step - accuracy: 0.6412 - loss: 1.0201 - val_accuracy: 0.6195 - val_loss: 1.0962
Epoch 5/15
591/591 ━━━━━━━━━━━━━━━━━━━━ 44s 74ms/step - accuracy: 0.6681 - loss: 0.9545 - val_accuracy: 0.6486 - val_loss: 1.0288
Epoch 6/15
591/591 ━━━━━━━━━━━━━━━━━━━━ 43s 72ms/step - accuracy: 0.6892 - loss: 0.8891 - val_accuracy: 0.6729 - val_loss: 0.9634
Epoch 7/15
591/591 ━━━━━━━━━━━━━━━━━━━━ 42s 72ms/step - accuracy: 0.7119 - loss: 0.8233 - val_accuracy: 0.6662 - val_loss: 0.9621
Epoch 8/15
591/591 ━━━━━━━━━━━━━━━━━━━━ 44s 74ms/step - accuracy: 0.7304 - loss: 0.7757 - 